# SSTW VideoShield formal reference Colab 入口

该 Notebook 负责单个 modern external baseline 的项目内官方流程闭环: 以同一 prompt / seed / attack runtime comparison unit 为锚点, 执行 source clone / resource build / official run / adapter 转换 / official bundle 落盘, 然后调用统一 external_baseline runner 将已完成的 official bundle 转写为 `metric_status: measured_formal` records。

Notebook 只调用仓库 helper, 不直接手写 records、tables、figures 或 reports。若其它 baseline 的 official bundle 尚未完成, 统一转写会为它们保留 governed unsupported rows, 不能替代正式 baseline 结果。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import subprocess
import sys
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = os.environ.get('SSTW_DRIVE_PROJECT_ROOT', '/content/drive/MyDrive/SSTW')
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 1.1 可编辑 workflow profile 切换
# 当前 baseline 参考 Notebook 默认面向 validation_scale。
# 如需复用到 pilot_paper, 可将 SSTW_WORKFLOW_PROFILE_VALUE 改为 'pilot_paper'。
import os

SSTW_WORKFLOW_PROFILE_VALUE = 'validation_scale'

if SSTW_WORKFLOW_PROFILE_VALUE.strip():
    os.environ['SSTW_WORKFLOW_PROFILE'] = SSTW_WORKFLOW_PROFILE_VALUE.strip()
    print('SSTW_WORKFLOW_PROFILE =', os.environ['SSTW_WORKFLOW_PROFILE'])
else:
    os.environ.pop('SSTW_WORKFLOW_PROFILE', None)
    print('SSTW_WORKFLOW_PROFILE 未显式设置。')


In [ ]:
# 2. 冷启动获取仓库代码
REPO_URL = os.environ.get('SSTW_REPO_URL', 'https://github.com/RICHAAARC/SSTW.git')
REPO_DIR = os.environ.get('SSTW_REPO_DIR', '/content/SSTW')
REPO_REF = os.environ.get('SSTW_REPO_REF', '').strip()

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 SSTW_REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"
if REPO_REF:
    !git fetch --all --tags
    !git checkout "$REPO_REF"
!git rev-parse --short HEAD


In [ ]:
# 3. 安装 baseline 参考运行的公共依赖
# baseline 官方仓库自身依赖仍应由 source intake / official command / native command 显式处理。
%pip install -U imageio imageio-ffmpeg av torchvision opencv-python gdown
extra_packages = os.environ.get('SSTW_BASELINE_EXTRA_PIP_PACKAGES', '').strip()
if extra_packages:
    !python -m pip install -U $extra_packages


In [ ]:
# 4. 可选 Hugging Face 认证
try:
    from huggingface_hub import login
except Exception:
    login = None

if os.environ.get('HF_TOKEN') and login is not None:
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided 或 huggingface_hub 未安装。')


In [ ]:
# 5. 运行 VideoShield 官方流程、official bundle 生成与统一 measured_formal 转写
# Notebook 只调用仓库 helper, 不在 cell 中手写 records、tables、figures 或 reports。
# helper 会以同一 prompt / seed / attack 协议为锚点生成 official bundle,
# 然后调用统一 external_baseline runner 转写 available bundle 为 measured_formal records。
BASELINE_ID = 'videoshield'
NOTEBOOK_ROLE = 'external_baseline_formal_scoring'
from paper_workflow.colab_utils.videoshield_formal_reference import run_default_videoshield_formal_reference_plan

result = run_default_videoshield_formal_reference_plan(repo_root=Path(os.environ.get('SSTW_REPO_DIR', '/content/SSTW')))
print(json.dumps(result, ensure_ascii=False, indent=2))

followup_status = {item.get('stage_id'): item.get('stage_status') for item in result.get('followup_results', [])}
print('followup_status =', json.dumps(followup_status, ensure_ascii=False, indent=2))

if result.get('formal_reference_decision') != 'PASS':
    raise RuntimeError(f"{BASELINE_ID} formal reference 未完成: {result.get('formal_reference_status')}")


In [ ]:
# 6. 可选治理审计
# 默认关闭, 避免每个 baseline Notebook 重复运行完整测试。正式提交前仍必须本地运行 pytest 与 harness。
if os.environ.get('SSTW_RUN_QUICK_TESTS_AFTER_BASELINE_REFERENCE', 'false').lower() == 'true':
    !pytest -q
    !python tools/harness/run_all_audits.py
